# 01 数据获取与清洗治理

**本笔记本作用**：完成《商业数据分析》期末大作业的数据获取与清洗环节，对应论文第 2 章。

- 数据源：Tableau 官方示例数据集 **Sample – Superstore**（官方 XLSX 一次性下载，**未使用任何爬虫**）
- 三张表：`Orders`（订单明细事实表）、`People`（区域经理维表）、`Returns`（退货维表）
- 产出：`data/clean/orders_clean.csv`、`data/clean/superstore.sqlite`、字段字典与清洗日志

> 合规说明见 `docs/数据来源合规声明.md`：数据来源、URL、获取方式、获取日期、授权范围均已登记。

## 0. 环境与路径准备

In [1]:
# ============================================================
# 运行环境：conda 环境 business_analysis | 内核 business_analysis
# 本单元格作用：把 scripts 目录加入路径，并提供按文件名加载脚本的工具函数
# ============================================================
import sys, pathlib, importlib.util

ROOT = pathlib.Path.cwd().parent          # 项目根目录（notebooks 的上一级）
sys.path.insert(0, str(ROOT / "scripts"))  # 让脚本里的 import common 生效

def load_step(module_name: str, filename: str):
    """按文件名加载 scripts 下的脚本（文件名以数字开头，不能直接 import）。"""
    p = ROOT / "scripts" / filename
    spec = importlib.util.spec_from_file_location(module_name, p)
    m = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(m)
    return m

print("项目根目录：", ROOT)

项目根目录： G:\ThinkPad\Documents\jupyter_wenjian\商业数据分析\数据分析大作业


## 1. 读取原始数据并做“数据体检”

只读原始文件（不覆盖），逐表统计行数、字段类型、缺失率、唯一值数，形成 `t01_数据体检报告.csv`。

In [2]:
load = load_step("step01", "01_load.py")
tables = load.run()
orders_raw = tables["Orders"]
orders_raw.head(3)

[1/3] 已读取原始文件：sample_-_superstore.xls
      - 工作表 Orders    规模  10194 行 × 21 列
      - 工作表 People    规模      4 行 ×  2 列
      - 工作表 Returns   规模    296 行 ×  2 列
[2/3] 数据体检


  [√] 已保存结果表：t01_数据体检报告.csv
  [√] 已保存结果表：t02_数据规模指标.csv
    指标           数值 单位
订单明细行数       10,194  行
 唯一订单数        5,111  单
 唯一客户数          804  位
 唯一产品数        1,862  个
 产品子类数           17  类
   城市数          542  个
  州/省数           59  个
  时间跨度         1457  天
 退货订单数          296  单
  销售总额 2,326,534.35 美元
  利润总额   292,296.81 美元
 整体毛利率       12.56%  %
[3/3] 导出字段字典
  [√] 已保存结果表：t03_字段字典.csv


      [√] 已生成 docs/数据来源合规声明.md


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country/Region,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,US-2023-103800,2023-01-03,2023-01-07,Standard Class,DP-13000,Darren Powers,Consumer,United States,Houston,...,77095,Central,OFF-PA-10000174,Office Supplies,Paper,"Message Book, Wirebound, Four 5 1/2"" X 4"" Form...",16.448,2,0.2,5.5512
1,2,US-2023-112326,2023-01-04,2023-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,60540,Central,OFF-BI-10004094,Office Supplies,Binders,GBC Standard Plastic Binding Systems Combs,3.540,2,0.8,-5.4870
2,3,US-2023-112326,2023-01-04,2023-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,60540,Central,OFF-LA-10003223,Office Supplies,Labels,Avery 508,11.784,3,0.2,4.2717


## 2. 字段字典

论文 2.1 节字段说明直接引用本表：字段名、中文含义、类型、业务解释。

In [3]:
load.data_dictionary()

,字段,中文名称,类型,业务含义
0,Row ID,行号,整数,记录唯一序号，1~10194
1,Order ID,订单号,字符串,订单主键，一单可含多行明细
2,Order Date,下单日期,日期,订单创建日期，2023-01-03 至 2026-12-30
3,Ship Date,发货日期,日期,订单发货日期，用于计算履约时长
4,Ship Mode,配送方式,类别,标准/二级/一级/当日达四种
5,Customer ID,客户编号,字符串,客户主键，共 804 位客户
6,Customer Name,客户姓名,字符串,客户名称
7,Segment,客户细分,类别,消费者/企业客户/家庭办公
8,Country/Region,国家/地区,类别,全量为美国
9,City,城市,字符串,542 个城市


## 3. 清洗与治理

清洗策略（每条都留痕到 `t04_清洗日志.csv`）：

| 问题 | 策略 | 依据 |
| --- | --- | --- |
| 文本首尾空格 | 全字段 strip | 防止主外键关联漏配 |
| 重复记录 | 按业务主键 `Row ID` 去重 | 一行明细唯一编号 |
| 日期类型 | `Order Date`/`Ship Date` 转 datetime | 支撑同比、履约天数计算 |
| 缺失值 | 数值→中位数；类别→众数；主键→剔除 | 中位数抗右偏与极端值 |
| 金额/数量非正 | 剔除 | 业务上不成立 |
| 发货早于下单 | 修正为下单日 +1 天 | 逻辑矛盾 |
| 折扣越界 | 裁剪到 [0, 0.8] | 业务折扣区间 |
| 销售额 3×IQR 极端值 | **标记但保留** | 大额订单是真实业务，删了会低估营收 |

In [4]:
clean = load_step("step02", "02_clean.py")
orders = clean.run()
orders.head(3)

[1/3] 已读取原始文件：sample_-_superstore.xls
      - 工作表 Orders    规模  10194 行 × 21 列
      - 工作表 People    规模      4 行 ×  2 列
      - 工作表 Returns   规模    296 行 ×  2 列


  [√] 已保存结果表：t04_清洗日志.csv

清洗日志：
         步骤   记录数 变动                                                                       说明
    0. 原始数据 10194  -                                                            读取 Orders 工作表
   1. 文本规范化 10194  0                                                 14 个文本字段去除首尾空格，避免主外键关联漏配
   2. 重复值处理 10194  0                                        整行重复 0 条、Row ID 重复 0 条（按业务主键保留首条）
  3. 日期类型统一 10194  0                 Order Date / Ship Date 转为 datetime，无法解析的记录剔除；邮编转字符串防丢前导零
   4. 缺失值处理 10194  0                             清洗前缺失单元 0 个（本例原始数据无缺失）；数值用中位数、类别用众数，主键缺失直接剔除
   5. 异常值处理 10194  0  销售额/销量非正剔除 0 条；发货早于下单修正 0 条；折扣越界裁剪 0 条；销售额 3×IQR 极端值标记 679 条（保留，真实大额订单）
    6. 指标派生 10194  0               新增 年份/月份/季度/年月、履约天数、毛利率、是否亏损、折扣金额、原价销售额、单件售价/利润 及 5 个中文维度列
  7. 关联退货维表 10194  0                    按 Order ID 左连 Returns：命中退货订单 296 个，覆盖率 5.8%（未命中视为未退货）
8. 关联区域经理维表 10194  0                                按 Region 左连 People：字段填充率 100.0%（4 个区域全覆盖）
   9. 主外键校验 10194  0 Order 

      [√] 已导出 data/clean/orders_clean.csv（10,194 行）


      [√] 已导出 SQLite：superstore.sqlite
  [√] 已保存结果表：t05_数据质量总览.csv

数据质量总览：
      检查项           结果
   清洗后记录数   10194.0000
      订单数    5111.0000
      客户数     804.0000
   剩余缺失单元       0.0000
销售额合计（美元） 2326534.3500
 利润合计（美元）  292296.8100
    整体毛利率       0.1256
   亏损明细占比       0.1865
   退货订单占比       0.0579


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country/Region,City,...,原价销售额,单件售价,单件利润,大区,客户细分_中文,品类,子类,配送方式,是否退货,区域经理
0,1,US-2023-103800,2023-01-03,2023-01-07,Standard Class,DP-13000,Darren Powers,Consumer,United States,Houston,...,20.56,8.224,2.7756,中部,消费者,办公用品,纸张,标准快递,0,Roxanne Rodriguez
1,2,US-2023-112326,2023-01-04,2023-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,17.70,1.770,-2.7435,中部,家庭办公,办公用品,装订用品,标准快递,0,Roxanne Rodriguez
2,3,US-2023-112326,2023-01-04,2023-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,14.73,3.928,1.4239,中部,家庭办公,办公用品,标签,标准快递,0,Roxanne Rodriguez


## 4. 清洗结果核验

- 记录数、订单数、客户数是否与原始一致（差异需能解释）
- 剩余缺失单元是否为 0
- 退货标记是否成功关联（覆盖率）
- 中文维度列是否齐全（图表全中文的前提）

In [5]:
print("记录数：", len(orders))
print("订单数：", orders["Order ID"].nunique())
print("客户数：", orders["Customer ID"].nunique())
print("剩余缺失：", int(orders.isna().sum().sum()))
print("中文维度列：", [c for c in orders.columns if c in ["大区","品类","子类","配送方式","客户细分_中文"]])

记录数： 10194
订单数： 5111
客户数： 804
剩余缺失： 0
中文维度列： ['大区', '客户细分_中文', '品类', '子类', '配送方式']


## 5. 异常与结构速览（为后续建模定锚）

看三件事：时间覆盖是否完整、品类结构是否长尾、亏损面有多大。

In [6]:
import pandas as pd
orders["年月"] = pd.to_datetime(orders["Order Date"]).dt.to_period("M").astype(str)
monthly = orders.groupby("年月")["Sales"].sum()
print("时间范围：", orders["Order Date"].min().date(), "→", orders["Order Date"].max().date())
print("月度均值（万美元）：", round(monthly.mean()/1e4, 2))
print("\n品类销售结构：")
print((orders.groupby("品类")["Sales"].sum()/orders["Sales"].sum()).map("{:.1%}".format))
print("\n亏损明细占比：", f"{orders['是否亏损'].mean():.1%}")
print("退货订单占比：", f"{orders.groupby('Order ID')['是否退货'].max().mean():.1%}")

时间范围： 2023-01-03 → 2026-12-30
月度均值（万美元）： 4.85

品类销售结构：
品类
办公用品    31.5%
家具      32.4%
技术产品    36.1%
Name: Sales, dtype: object

亏损明细占比： 18.6%
退货订单占比： 5.8%


## 6. 导出与复用

同时导出 CSV（便于 Excel 核对）与 SQLite（便于 SQL 复现），满足“可导出复用”的评分点。

In [7]:
import os
for f in ["orders_clean.csv", "dim_people.csv", "dim_returns.csv", "superstore.sqlite"]:
    p = ROOT / "data" / "clean" / f
    print(f"{f:<22} {p.stat().st_size/1024:>8.1f} KB  exists={p.exists()}")

orders_clean.csv         3802.4 KB  exists=True
dim_people.csv              0.1 KB  exists=True
dim_returns.csv             5.8 KB  exists=True
superstore.sqlite        3940.0 KB  exists=True
